# H&M 2년 M4 K=1 seed 44 — 병렬 arm 최종 집계

M1·실제 M4·순열 M4 세 arm이 모두 100 epoch를 완료한 뒤에만 실행합니다. 저장된 세 결과를 불러와 비교표와 귀속 판정을 생성합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
import subprocess
import sys

REVIEWED_SHA = '3188b01c360d54e78338eaebdf2a8ecb6a6d52d8'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
REPO_DIR = Path('/content/clv-m2-lightgcn-runner-hm-m4-k1-seed44-aggregate')

if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'기존 경로가 Git 저장소가 아닙니다: {REPO_DIR}')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '-q', 'origin', REVIEWED_SHA], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-q', '--detach', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('검토된 코드:', actual_sha)


In [ ]:
import json
import torch
import lightgcn_clv_m4_k1_assignment_control_hm2y as hm_screen

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert hm_screen.CODE_VERSION == 'm4-personalized-positive-weight-k1-assignment-control-hm2y-development-screen-v1'
cfg = hm_screen.configure_hm2y_m4_assignment_screen(seed=44, shuffle_seed=44)
summary = hm_screen.preflight_summary(cfg)
assert summary['dataset'] == 'hm'
assert summary['seed'] == 44 and cfg.shuffle_seed == 44
assert summary['staged_replication']['seeds'] == [42, 43, 44]
assert summary['trained_models'] == list(hm_screen.MODEL_IDS)
assert summary['fixed']['negative_count'] == 1
assert summary['fixed']['batch_size'] == 131072
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
assert summary['checkpointing']['save_after_each_completed_epoch'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\n중단 후 재실행하면 마지막으로 완료된 epoch 다음부터 이어집니다.')


In [ ]:
# 세 arm이 모두 완료된 뒤에만 실행하세요. 완료 arm은 재학습하지 않고 불러옵니다.
result_df = hm_screen.run_hm2y_m4_assignment_screen(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

core = [
    'model_id', 'm4_assignment', 'row_weight_cv',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10', 'vndcg@10',
    'coverage@10', 'user_value_tendency_recommended_price_alignment',
]
print('1) 핵심 절대지표')
show(result_df[[column for column in core if column in result_df.columns]])
print('2) 전체 비교지표')
show(result_df.attrs['comparison'])
print('3) Top-10 변경 비율')
show(result_df.attrs['top10_overlap'])
print('4) 사전 고정 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('5) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
